In [1]:
import pandas as pd

split_path = '/kaggle/input/mooccubex-data-cleaned/split_data/raw_3'
train_pl = pd.read_csv(f'{split_path}/train.csv')
test_1_pl = pd.read_csv(f'{split_path}/test_p1.csv')
test_2_pl = pd.read_csv(f'{split_path}/test_p2.csv')
test_3_pl = pd.read_csv(f'{split_path}/test_p3.csv')
test_4_pl = pd.read_csv(f'{split_path}/test_p4.csv')
val = pd.read_csv(f'{split_path}/val_p4.csv')

print("Đã load xong toàn bộ dữ liệu train / val / test.")

Đã load xong toàn bộ dữ liệu train / val / test.


In [2]:
import pandas as pd
import polars as pl

print('\nKiểm tra các cột không phải kiểu số trong train_pl:')
string_columns = []
for col in train_pl.columns:
    if not pd.api.types.is_numeric_dtype(train_pl[col].dtype):
        print(f"Cột '{col}' có kiểu dữ liệu: {train_pl[col].dtype}")
        string_columns.append(col)

if not string_columns:
    print("Không có cột non-numeric cần xử lý.")
else:
    print(f"Các cột non-numeric: {string_columns} (không áp dụng Mean Imputation)")


Kiểm tra các cột không phải kiểu số trong train_pl:
Không có cột non-numeric cần xử lý.


In [3]:
mean_values = {}
for col in train_pl.columns:
    if pd.api.types.is_numeric_dtype(train_pl[col]):
        mean_values[col] = train_pl[col].mean()

print(f"\n Đã tính MEAN cho {len(mean_values)} cột số từ tập TRAIN.")


 Đã tính MEAN cho 182 cột số từ tập TRAIN.


In [4]:
import pandas as pd # Ensure pandas is imported if not already
# import polars as pl # Polars is not needed for this corrected cell

dataframes_for_imputation = [
    (train_pl, 'train_pl'),
    (test_1_pl, 'test_1_pl'),
    (test_2_pl, 'test_2_pl'),
    (test_3_pl, 'test_3_pl'),
    (test_4_pl, 'test_4_pl'),
    (val, 'val')
]

for df_obj, df_name in dataframes_for_imputation:
    print(f"\n Đang điền khuyết cho {df_name}...")

    missing_before = df_obj.isnull().sum().sum()

    columns_to_impute = [
        col for col in df_obj.columns
        if pd.api.types.is_numeric_dtype(df_obj[col].dtype)
        and df_obj[col].isnull().any()
        and col in mean_values
    ]

    if columns_to_impute:
        means_for_current_df = {col: mean_values[col] for col in columns_to_impute}
        df_obj = df_obj.fillna(means_for_current_df)
        print(f"Đã điền khuyết bằng MEAN cho {len(columns_to_impute)} cột.")
    else:
        print("Không có cột số nào cần điền khuyết.")

    missing_after = df_obj.isnull().sum().sum()
    print(f"Số giá trị khuyết: trước = {missing_before}, sau = {missing_after}")
     # Cập nhật lại biến gốc
    if df_name == 'train_pl':
        train_pl = df_obj
    elif df_name == 'test_1_pl':
        test_1_pl = df_obj
    elif df_name == 'test_2_pl':
        test_2_pl = df_obj
    elif df_name == 'test_3_pl':
        test_3_pl = df_obj
    elif df_name == 'test_4_pl':
        test_4_pl = df_obj
    elif df_name == 'val':
        val = df_obj

print("\n HOÀN TẤT QUÁ TRÌNH ĐIỀN KHUYẾT DỮ LIỆU BẰNG MEAN IMPUTATION.")


 Đang điền khuyết cho train_pl...
Đã điền khuyết bằng MEAN cho 156 cột.
Số giá trị khuyết: trước = 284115606, sau = 0

 Đang điền khuyết cho test_1_pl...
Đã điền khuyết bằng MEAN cho 39 cột.
Số giá trị khuyết: trước = 8618295, sau = 0

 Đang điền khuyết cho test_2_pl...
Đã điền khuyết bằng MEAN cho 78 cột.
Số giá trị khuyết: trước = 17487288, sau = 0

 Đang điền khuyết cho test_3_pl...
Đã điền khuyết bằng MEAN cho 117 cột.
Số giá trị khuyết: trước = 26466434, sau = 0

 Đang điền khuyết cho test_4_pl...
Đã điền khuyết bằng MEAN cho 156 cột.
Số giá trị khuyết: trước = 35508014, sau = 0

 Đang điền khuyết cho val...
Đã điền khuyết bằng MEAN cho 156 cột.
Số giá trị khuyết: trước = 35524635, sau = 0

 HOÀN TẤT QUÁ TRÌNH ĐIỀN KHUYẾT DỮ LIỆU BẰNG MEAN IMPUTATION.


In [5]:
all_datasets = [
    (train_pl, 'train'),
    (val, 'val'),
    (test_1_pl, 'test_1'),
    (test_2_pl, 'test_2'),
    (test_3_pl, 'test_3'),
    (test_4_pl, 'test_4')
]

for df_obj, df_name in all_datasets:
    print(f"\n Xử lý dataset {df_name}...")

    y = df_obj['label_3']
    cols_to_drop = ['user_id', 'course_id', 'label_3']
    X = df_obj.drop(columns=cols_to_drop)

    globals()[f'X_{df_name}'] = X
    globals()[f'y_{df_name}'] = y

    print(f"X_{df_name}: {X.shape}, y_{df_name}: {y.shape}")

print("\n Hoàn tất tách feature và label.")


 Xử lý dataset train...
X_train: (1859619, 179), y_train: (1859619,)

 Xử lý dataset val...
X_val: (232452, 179), y_val: (232452,)

 Xử lý dataset test_1...
X_test_1: (232453, 179), y_test_1: (232453,)

 Xử lý dataset test_2...
X_test_2: (232453, 179), y_test_2: (232453,)

 Xử lý dataset test_3...
X_test_3: (232453, 179), y_test_3: (232453,)

 Xử lý dataset test_4...
X_test_4: (232453, 179), y_test_4: (232453,)

 Hoàn tất tách feature và label.


In [6]:
# TRAIN MODEL (LINEAR REGRESSION – BASELINE)
from sklearn.linear_model import LinearRegression

model = LinearRegression()
print("\n Bắt đầu huấn luyện mô hình...")
model.fit(X_train, y_train)
print("Huấn luyện xong mô hình.")


 Bắt đầu huấn luyện mô hình...
Huấn luyện xong mô hình.


In [7]:
# DỰ ĐOÁN
print("\n Đang dự đoán trên val và test...")

y_pred_val = model.predict(X_val)
y_pred_test_1 = model.predict(X_test_1)
y_pred_test_2 = model.predict(X_test_2)
y_pred_test_3 = model.predict(X_test_3)
y_pred_test_4 = model.predict(X_test_4)

print("Dự đoán hoàn tất.")


 Đang dự đoán trên val và test...
Dự đoán hoàn tất.


In [8]:
# ĐÁNH GIÁ ACCURACY
from sklearn.metrics import accuracy_score
import numpy as np

y_pred_val_rounded = np.round(y_pred_val)
y_pred_test_1_rounded = np.round(y_pred_test_1)
y_pred_test_2_rounded = np.round(y_pred_test_2)
y_pred_test_3_rounded = np.round(y_pred_test_3)
y_pred_test_4_rounded = np.round(y_pred_test_4)

print("\n Accuracy:")
print('Validation:', accuracy_score(y_val, y_pred_val_rounded))
print('Test 1:', accuracy_score(y_test_1, y_pred_test_1_rounded))
print('Test 2:', accuracy_score(y_test_2, y_pred_test_2_rounded))
print('Test 3:', accuracy_score(y_test_3, y_pred_test_3_rounded))
print('Test 4:', accuracy_score(y_test_4, y_pred_test_4_rounded))



 Accuracy:
Validation: 0.9958012837058834
Test 1: 0.0
Test 2: 0.0
Test 3: 0.0020950471708259303
Test 4: 0.9958271134379854


In [9]:
X_train

,course_duration,start_year,start_month,start_day,enrollment_to_end,gender,num_course_order,avg_interval_enroll_time,max_interval_enroll_time,min_interval_enroll_time,...,cmt_p4_max_comment_length_phase4,cmt_p4_text_diversity_phase4,cmt_p4_comment_days_active_phase4,cmt_p4_entropy_time_mean_phase4,cmt_p4_most_common_time_bin_phase4,cmt_p4_time_entropy_var_phase4,cmt_p4_positive_ratio_phase4,cmt_p4_neutral_ratio_phase4,cmt_p4_negative_ratio_phase4,cmt_p4_sentiment_entropy_phase4
0,176,2020,2,6,170,2.0,6,2.058000,9.56,0.00,...,37.895714,151.214508,1.305472,0.633849,29.86416,0.009356,0.241014,0.685691,0.073295,0.28781
1,114,2020,10,30,76,0.0,58,7.192982,59.22,0.02,...,37.895714,151.214508,1.305472,0.633849,29.86416,0.009356,0.241014,0.685691,0.073295,0.28781
2,170,2020,2,12,69,0.0,56,7.414727,51.91,0.19,...,37.895714,151.214508,1.305472,0.633849,29.86416,0.009356,0.241014,0.685691,0.073295,0.28781
3,151,2020,2,16,138,2.0,7,7.661667,34.84,0.01,...,37.895714,151.214508,1.305472,0.633849,29.86416,0.009356,0.241014,0.685691,0.073295,0.28781
4,173,2020,10,30,140,2.0,2,71.060000,71.06,71.06,...,37.895714,151.214508,1.305472,0.633849,29.86416,0.009356,0.241014,0.685691,0.073295,0.28781
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1859614,91,2019,12,31,52,0.0,49,7.943542,33.74,0.12,...,37.895714,151.214508,1.305472,0.633849,29.86416,0.009356,0.241014,0.685691,0.073295,0.28781
1859615,121,2020,9,1,31,0.0,60,6.899322,55.12,0.02,...,37.895714,151.214508,1.305472,0.633849,29.86416,0.009356,0.241014,0.685691,0.073295,0.28781
1859616,93,2020,4,13,91,2.0,2,63.480000,63.48,63.48,...,37.895714,151.214508,1.305472,0.633849,29.86416,0.009356,0.241014,0.685691,0.073295,0.28781
1859617,121,2020,9,1,37,1.0,3,0.025000,0.05,0.00,...,37.895714,151.214508,1.305472,0.633849,29.86416,0.009356,0.241014,0.685691,0.073295,0.28781


In [10]:
import os

# Thư mục output chuẩn của Kaggle
output_path = "/kaggle/working/mean_imputation"
os.makedirs(output_path, exist_ok=True)

# Lưu các tập dữ liệu
train_pl.to_csv(f"{output_path}/train.csv", index=False)
val.to_csv(f"{output_path}/val.csv", index=False)
test_1_pl.to_csv(f"{output_path}/test_1.csv", index=False)
test_2_pl.to_csv(f"{output_path}/test_2.csv", index=False)
test_3_pl.to_csv(f"{output_path}/test_3.csv", index=False)
test_4_pl.to_csv(f"{output_path}/test_4.csv", index=False)

print(f"\n Đã lưu dữ liệu sau Mean Imputation tại:\n{output_path}")



 Đã lưu dữ liệu sau Mean Imputation tại:
/kaggle/working/mean_imputation
